<a href="https://colab.research.google.com/github/spicecat/unhash/blob/main/numba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[https://scratch.mit.edu/projects/164028530](https://scratch.mit.edu/projects/164028530)

In [1]:
# @title
%pip install ruff -q

from IPython.core.magic import register_cell_magic
import subprocess
import tempfile
import os

@register_cell_magic
def ruff_format(line, cell):
    """Cell magic to format code with ruff. Usage: %%ruff_format"""
    with tempfile.NamedTemporaryFile(mode='w+', delete=False, suffix='.py') as f:
        f.write(cell)
        file_path = f.name
    try:
        subprocess.run(['ruff', 'format', file_path], check=True, capture_output=True, text=True)
        with open(file_path, 'r') as f:
            print(f.read())
    except subprocess.CalledProcessError as e:
        print("Error during formatting:", e.stderr)
    finally:
        os.remove(file_path)

@register_cell_magic
def ruff_lint(line, cell):
    """Cell magic to lint code with ruff. Usage: %%ruff_lint"""
    with tempfile.NamedTemporaryFile(mode='w+', delete=False, suffix='.py') as f:
        f.write(cell)
        file_path = f.name
    try:
        result = subprocess.run(['ruff', 'check', file_path], capture_output=True, text=True)
        if result.stdout:
            print(result.stdout)
        else:
            print("No issues found.")
    finally:
        os.remove(file_path)

In [2]:
# @title
# %conda install --quiet numpy numba
import numpy as np
from numpy.typing import NDArray
import numba as nb

np.set_printoptions(formatter={'int':hex})

In [3]:
Char = np.uint8
NChar = nb.types.uint8
Enc = NDArray[Char]

H = Char(2)
HRange = np.arange(H, dtype=Char)
CW = Char(np.iinfo(Char).bits // H)
C = 1 << CW
CHARS = "abcdefghijklmnopqrstuvwxyz0123456789 "[:C]


@nb.jit(NChar[:](nb.types.string))
def encode(s: str) -> Enc:
    enc = np.array([+CHARS.index(c) for c in s], dtype=Char)
    return (enc.reshape(-1, H) << CW * HRange).sum(axis=1, dtype=Char)


@nb.jit(nb.types.string(NChar[:]))
def decode(e: Enc):
    enc = (
        e.repeat(H).reshape(-1, H) >> CW * HRange & C - 1
    ).flatten()
    return "".join([CHARS[c] for c in enc])


del HRange, CW, CHARS

# s = "aaaaddddabcdabcd"
s = "apabcdef"
enc = encode(s)
print(f"{H=} {C=} {enc} {decode(enc)}")
del s, enc

H=np.uint8(2) C=np.uint8(16) [0xf0 0x10 0x32 0x54] apabcdef


In [4]:
Con = np.uint16
NCon = nb.types.uint16
Cons = NDArray[Con]

MQ = np.float64(np.iinfo(Con).max + 1)
M = np.uint64(100_000)
Q = MQ / M

S = Char(4)
N = Char(16)
assert N % H == 0
P0 = np.array([11, 17, 7, 5], np.uint64)
P1 = np.array([29, 31, 17, 13], np.uint16)
P2 = np.array([53, 67, 103, 47], np.uint16)
P3 = np.array([52, 12, 24, 30], np.uint16)
P4 = np.array([0, 90, 0, 90], np.uint64)
idx = np.arange(N, dtype=np.uint64)
A = np.outer(idx + 1, P0) % P1 + P2
B = A * P3 + P4
E = np.outer(np.arange(C, dtype=Char), np.ones(N, dtype=Char))[:, :, np.newaxis]
angle = A * E + B
sin = 5 * np.sin(np.radians(angle))
CTABLE = np.round((sin - np.floor(sin)) * MQ).astype(Con)
HTABLE: Cons = (
    CTABLE.reshape(C, N // H, H, S)
    .transpose(2, 0, 1, 3)[
        np.arange(H, dtype=Char).reshape((H,) + (1,) * H),
        np.indices((C.tolist(),) * H)[::-1],
    ]
    .sum(axis=0, dtype=Con)
    .reshape(-1, N // H, S)
)


@nb.guvectorize([(NChar[:], NCon[:, :, :], NCon[:])], "(l),(c,n,s)->(s)")
def hashify(enc: Enc, htable: Cons, cons: Cons):
    for i in range(len(enc)):
        cons += htable[enc[i], i]


del MQ, M, P0, P1, P2, P3, P4, idx, A, B, E, angle, sin, CTABLE

s = ["apabcdef", "mkkaphmi", "pckkemhj", "fhgkgbhl"]
enc = np.array(list(map(encode, s)))
cons = np.zeros((len(enc), S), dtype=Con)
hashify(enc, HTABLE, cons)
print(cons)
del s, enc, cons

[[0xb88c 0x8bd4 0xef8e 0x9f26]
 [0x499b 0x301e 0x2bca 0xc5bd]
 [0xed63 0x93ae 0xa66 0xd9a5]
 [0x7da1 0x30cc 0xc30 0x5552]]


In [5]:
# @title
W = 4
NT = 100_000_000
np.random.seed(0)
enc = np.random.randint(0, np.iinfo(Char).max, NT * W, dtype=Char).reshape(-1, W)

cons = np.empty((NT, S), dtype=Con)
%timeit hashify(enc, HTABLE, cons)
del W, NT, enc, cons
# 16.2 ms ± 1.31 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)
# 3.57 s ± 1.02 s per loop (mean ± std. dev. of 7 runs, 1 loop each)
# 15 ms ± 1.45 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)
# 15.7 ms ± 1.47 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)

3.57 s ± 1.02 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
# %pip install numba-cuda
from numba import cuda
cuda.config.CUDA_ENABLE_PYNVJITLINK = True

Con = np.uint16
NCon = nb.types.uint16
Cons = NDArray[Con]

MQ = np.float64(np.iinfo(Con).max + 1)
M = np.uint64(100_000)
Q = MQ / M

S = Char(4)
N = Char(16)
assert N % H == 0
P0 = np.array([11, 17, 7, 5], np.uint64)
P1 = np.array([29, 31, 17, 13], np.uint16)
P2 = np.array([53, 67, 103, 47], np.uint16)
P3 = np.array([52, 12, 24, 30], np.uint16)
P4 = np.array([0, 90, 0, 90], np.uint64)
idx = np.arange(N, dtype=np.uint64)
A = np.outer(idx + 1, P0) % P1 + P2
B = A * P3 + P4
E = np.outer(np.arange(C, dtype=Char), np.ones(N, dtype=Char))[:, :, np.newaxis]
angle = A * E + B
sin = 5 * np.sin(np.radians(angle))
CTABLE = np.round((sin - np.floor(sin)) * MQ).astype(Con)
HTABLE: Cons = (
    CTABLE.reshape(C, N // H, H, S)
    .transpose(2, 0, 1, 3)[
        np.arange(H, dtype=Char).reshape((H,) + (1,) * H),
        np.indices((C.tolist(),) * H)[::-1],
    ]
    .sum(axis=0, dtype=Con)
    .reshape(-1, N // H, S)
)


@cuda.jit([(NChar[:,:], NCon[:, :, :], NCon[:, :])])
def cu_hashify(enc: Enc, htable: Cons, cons: Cons):
    tid = cuda.grid(1)
    if tid >= len(cons):
        return
    for i in range(enc.shape[1]):
        for s in range(S):
            cons[tid, s] += htable[enc[tid, i], i, s]


del MQ, M, P0, P1, P2, P3, P4, idx, A, B, E, angle, sin, CTABLE

s = ["apabcdef", "mkkaphmi", "pckkemhj", "fhgkgbhl"]
enc = np.array(list(map(encode, s)))
cu_enc = cuda.to_device(enc)
cu_cons = cuda.to_device(np.zeros((len(enc), S), dtype=Con))
cu_htable = cuda.to_device(HTABLE)
cu_hashify.forall(len(s))(cu_enc, cu_htable, cu_cons)
print(cu_cons.copy_to_host())
del s, enc, cu_enc, cu_cons

[[0xb88c 0x8bd4 0xef8e 0x9f26]
 [0x499b 0x301e 0x2bca 0xc5bd]
 [0xed63 0x93ae 0xa66 0xd9a5]
 [0x7da1 0x30cc 0xc30 0x5552]]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:680: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [19]:
W = 4
NT = 100_000_000
np.random.seed(0)
enc = np.random.randint(0, np.iinfo(Char).max, NT * W, dtype=Char).reshape(-1, W)

cu_enc = cuda.to_device(enc)
cu_cons = cuda.to_device(np.zeros((NT, S), dtype=Con))
cu_htable = cuda.to_device(HTABLE)
%timeit cu_hashify.forall(NT)(cu_enc, cu_htable, cu_cons)

del W, NT, enc, cu_enc, cu_cons, cu_htable
# 164 µs ± 90.4 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
# @title
Key = np.uint64
Keys = NDArray[Key]
NKey = nb.types.uint64

ERR = Con(N // 2.5)
L = np.iinfo(Con).bits
SRange = np.arange(S, dtype=np.uint16)
SW = np.iinfo(Key).bits // Key(S)
SShift = (S - 1) * SW - SRange * L % np.iinfo(Key).bits


@nb.guvectorize([(NCon[:], NKey[:])], "(s)->()")
def pack(cons: Cons, key: Keys):
    # key[0] = (
    #     (
    #         (cons // ERR).view(Key)[SRange * L // np.iinfo(Key).bits] >> SShift
    #         & Key((1 << SW) - 1)
    #     )
    #     << SW * SRange
    # ).sum(dtype=Key)
    # key[:] = (cons >> ERR).view(Key)
    temp_key = Key(0)
    for i in range(len(cons)):
        temp_key |= cons[i] >> ERR << i * np.iinfo(Key).bits // S
    key[0] = temp_key


del ERR, L, SRange, SW, SShift

s = ["apabcdef", "mkkaphmi", "pckkemhj", "fhgkgbhl"]
enc = np.array(list(map(encode, s)))
cons = np.empty((len(enc), S), dtype=Con)
hashify(enc, HTABLE, cons)
T = np.array(
    [
        [72090, 54618, 93575, 62167],
        [28752, 18794, 17102, 77239],
        [92729, 57686, 4061, 85015],
        [49071, 19061, 4760, 33326]
    ],
    dtype=Key,
)
keys = np.empty(len(T), dtype=Key)
pack(cons, keys)
target_key = np.empty(len(T), dtype=Key)
pack((T * Q).astype(Con), target_key)
diff = keys - target_key
print(cons)
print(keys)
print(diff)
del s, enc, cons, T, keys, target_key, diff


In [ ]:
# @title
W = 4
NT = 100_000_000
np.random.seed(0)
enc = np.random.randint(0, np.iinfo(Char).max, NT * W, dtype=Char).reshape(-1, W)

cons = np.empty((NT, S), dtype=Con)
hashify(enc, HTABLE, cons)
keys = np.empty(NT, dtype=Key)
%timeit pack(cons, keys)
del W, NT, enc, cons, keys
# 497 ms ± 8.45 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
# %pip install numba-cuda
from numba import cuda
cuda.config.CUDA_ENABLE_PYNVJITLINK = True

Key = np.uint64
Keys = NDArray[Key]
NKey = nb.types.uint64

ERR = Con(N // 2.5)

@cuda.jit([(NCon[:, :], NKey[:])])
def cu_pack(cons: Cons, key: Keys):
    tid = cuda.grid(1)
    temp_key = Key(0)
    for i in range(4):
        temp_key |= cons[tid, i] >> ERR << i * np.iinfo(Key).bits // S
    key[tid] = temp_key

s = ["apabcdef", "mkkaphmi", "pckkemhj", "fhgkgbhl"]
enc = np.array(list(map(encode, s)))
cons = np.empty((len(enc), S), dtype=Con)
hashify(enc, HTABLE, cons)
T = np.array(
    [
        [72090, 54618, 93575, 62167],
        [28752, 18794, 17102, 77239],
        [92729, 57686, 4061, 85015],
        [49071, 19061, 4760, 33326]
    ],
    dtype=Key,
)
cu_cons = cuda.to_device(cons)
cu_keys = cuda.to_device(np.empty(len(T), dtype=Key))
cu_pack.forall(len(T))(cu_cons, cu_keys)
print(cu_keys.copy_to_host())

del s, enc, cons, T, cu_cons, cu_keys

In [ ]:
W = 4
NT = 100_000_000
np.random.seed(0)
enc = np.random.randint(0, np.iinfo(Char).max, NT * W, dtype=Char).reshape(-1, W)

cons = np.empty((NT, S), dtype=Con)
hashify(enc, HTABLE, cons)
cu_cons = cuda.to_device(cons)
cu_keys = cuda.to_device(np.zeros(NT, dtype=Key))
%timeit cu_pack.forall(NT)(cu_cons, cu_keys)

del W, NT, enc, cons, cu_cons, cu_keys
# 6.45 ms ± 6.16 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [ ]:
# @title
import numpy as np
import numba as nb

np.random.seed(0)

Key = np.uint64
NKey = nb.types.uint64
Keys = NDArray[Key]
NKeys = NKey[:]

# size = 37 ** 6
size = 1_000_000
print(f"Generating {size} unique random keys...")
keys = np.random.randint(0, np.iinfo(Key).max, size, dtype=Key)
assert np.unique(keys).size == size

In [ ]:
# @title
segment_length = 1 << int(np.log(size) / np.log(3.33) + 2.25)
segment_length_mask = segment_length - 1

arity = 3
total_segments = (int(size * 1.125) + segment_length_mask) // segment_length
segment_count = total_segments - (arity - 1)
segment_count_length = segment_count * segment_length


@nb.jit(nb.types.containers.UniTuple(nb.types.uint32, arity)(NKey))
def _get_hashes(key: Key):
    h0 = (key >> 32) * segment_count_length + (
        (key & 0xFFFFFFFF) * segment_count_length >> 32
    ) >> 32
    h1 = h0 + segment_length ^ (key >> 18) & segment_length_mask
    h2 = h0 + 2 * segment_length ^ key & segment_length_mask
    return h0, h1, h2


@nb.jit(NKeys(NKeys))
def populate(keys: Keys):
    block_bits = 1
    while (1 << block_bits) < segment_count:
        block_bits += 1
    block = 1 << block_bits
    start_pos = np.zeros(block, dtype=np.uint32)

    for i in range(block):
        start_pos[i] = np.uint32((i * size) >> block_bits)

    reverse_order = np.zeros(size + 1, dtype=Key)
    reverse_order.fill(0)
    reverse_order[size] = 1
    mask_block = Key(block - 1)
    for h in keys:
        segment_index = h >> (64 - block_bits)
        while reverse_order[start_pos[segment_index]] != 0:
            segment_index = (segment_index + Key(1)) & mask_block
        reverse_order[start_pos[segment_index]] = h
        start_pos[segment_index] += 1

    capacity = total_segments * segment_length
    t2count = np.zeros(capacity, dtype=np.uint8)
    t2hash = np.zeros(capacity, dtype=Key)
    for i in range(size):
        hash_val = reverse_order[i]
        h0, h1, h2 = _get_hashes(hash_val)
        t2count[h0] += 4
        t2hash[h0] ^= hash_val
        t2count[h1] += 4
        t2count[h1] ^= 1
        t2hash[h1] ^= hash_val
        t2count[h2] += 4
        t2count[h2] ^= 2
        t2hash[h2] ^= hash_val

    alone = np.zeros(capacity, dtype=np.uint32)
    q_size = 0
    for i in range(capacity):
        if (t2count[i] >> 2) == 1:
            alone[q_size] = i
            q_size += 1

    reverse_h = np.zeros(size, dtype=np.uint8)
    stack_size = 0
    while q_size > 0:
        q_size -= 1
        index = np.uint32(alone[q_size])
        if (t2count[index] >> 2) == 1:
            hash_val = t2hash[index]
            found = t2count[index] & 3
            reverse_h[stack_size] = found
            reverse_order[stack_size] = hash_val
            stack_size += 1
            h_all = _get_hashes(hash_val)

            other_index1 = h_all[(found + 1) % 3]
            if (t2count[other_index1] >> 2) == 2:
                alone[q_size] = other_index1
                q_size += 1
            t2count[other_index1] -= 4
            t2count[other_index1] ^= (found + 1) % 3
            t2hash[other_index1] ^= hash_val

            other_index2 = h_all[(found + 2) % 3]
            if (t2count[other_index2] >> 2) == 2:
                alone[q_size] = other_index2
                q_size += 1
            t2count[other_index2] -= 4
            t2count[other_index2] ^= (found + 2) % 3
            t2hash[other_index2] ^= hash_val

    fingerprints = np.zeros(capacity, dtype=Key)
    for i in range(size - 1, -1, -1):
        hash_val = reverse_order[i]
        found = reverse_h[i]
        h_all = _get_hashes(hash_val)
        fingerprints[h_all[found]] = (
            hash_val
            ^ fingerprints[h_all[(found + 1) % 3]]
            ^ fingerprints[h_all[(found + 2) % 3]]
        )
    return fingerprints

del arity, total_segments, segment_count

print("Populating filter...")
fingerprints = populate(keys)
print("Filter populated.")
np.save("fingerprints.npy", fingerprints)


In [ ]:
# @title
%timeit populate(keys)
# 78.8 ms ± 1.99 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)

In [ ]:
# @title
fingerprints = np.load('fingerprints.npy')

@nb.vectorize([nb.types.boolean(NKey)], identity=0)
def contain(key):
    h0 = (key >> 32) * segment_count_length + ((key & 0xFFFFFFFF) * segment_count_length >> 32) >> 32
    h1 = h0 + segment_length ^ (key >> 18) & segment_length_mask
    h2 = h0 + 2 * segment_length ^ key & segment_length_mask
    # h0, h1, h2 = _get_hashes(key)
    return (key ^ fingerprints[h0] ^ fingerprints[h1] ^ fingerprints[h2]) == 0

found_count = contain(keys).sum()
print(f"Found {found_count}/{size} of the original keys.")
assert found_count == size

num_test_keys = 100000
test_keys = np.random.randint(0, np.iinfo(Key).max, size=num_test_keys, dtype=Key)
false_positives = contain(test_keys).sum()
print(f"False positive rate: {false_positives / num_test_keys:.4f}")
assert false_positives / num_test_keys < 0.01

np.random.seed(1)
num_test_keys = 10_000_000
test_keys = np.random.randint(0, np.iinfo(Key).max, num_test_keys, dtype=Key)
%timeit contain(test_keys)
# 571 ms ± 301 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
# 99.6 ms ± 1.89 ms per loop (mean ± std. dev. of 7 runs, 10 loops each) GPU

In [ ]:
# %pip install numba-cuda
import numpy as np
import numba as nb
from numba import cuda
cuda.config.CUDA_ENABLE_PYNVJITLINK = True

Key = np.uint64
NKey = nb.types.uint64
Keys = NDArray[Key]
NKeys = NKey[:]

fingerprints = np.load('fingerprints.npy')
size = 1_000_000

segment_length = 1 << int(np.log(size) / np.log(3.33) + 2.25)
segment_length_mask = segment_length - 1

arity = 3
total_segments = (int(size * 1.125) + segment_length_mask) // segment_length
segment_count = total_segments - (arity - 1)
segment_count_length = segment_count * segment_length
capacity = total_segments * segment_length

num_test_keys = 1_000_000

@cuda.jit
def cu_contain(results: Keys, fingerprints: Keys, keys: Keys):
    tid = cuda.grid(1)
    if tid < num_test_keys:
        h0 = (keys[tid] >> 32) * segment_count_length + ((keys[tid] & 0xFFFFFFFF) * segment_count_length >> 32) >> 32
        h1 = h0 + segment_length ^ (keys[tid] >> 18) & segment_length_mask
        h2 = h0 + 2 * segment_length ^ keys[tid] & segment_length_mask
        results[tid] = (keys[tid] ^ fingerprints[h0] ^ fingerprints[h1] ^ fingerprints[h2]) == 0

F = cuda.to_device(fingerprints)
R = cuda.to_device(np.zeros(num_test_keys, dtype=Key))
np.random.seed(0)
K = cuda.to_device(np.random.randint(0, np.iinfo(Key).max, num_test_keys, dtype=Key))

cu_contain.forall(size)(R, F, K)
R.copy_to_host().sum()

In [ ]:
%timeit cu_contain.forall(size)(R, F, K)
# 894 µs ± 69.4 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)

In [ ]:
import numpy as np
import numba as nb

L = 5

np.random.seed(42)
target = np.random.randint(0, C, L)


@nb.jit
def check(a: np.ndarray):
    return np.array_equal(a, target)


@nb.jit(parallel=True)
def search():
    done = False
    res = np.zeros(L, dtype=Con)
    for i in nb.prange(C):
        if done:
            continue
        enc = np.full(L, C - 1, dtype=Con)
        enc[0] = i
        while True:
            if check(enc):
                done = True
                for k in range(N):
                    res[k] = enc[k]
                break
            j = L - 1
            while j > 0:
                if enc[j]:
                    enc[j] -= 1
                    break
                else:
                    enc[j] = C - 1
                    j -= 1
            else:
                break
    return res


# search()


In [ ]:
%timeit search()

In [ ]:
import numpy as np
import numba as nb

@nb.njit(parallel=True)
def generate_first_m_combinations(M, N=32, L=6):
    mask = 31
    shift_amount = 4
    result = np.empty((M, L), dtype=np.int64)
    for i in nb.prange(M):
        val = i + 0
        for j in range(L):
            result[i, j] = val & mask
            val >>= shift_amount
    return result

N = 32
L = 6
M = 100_000_000
# first_combinations = generate_first_m_combinations(M, N, L)
# first_combinations.nbytes / (1 << 30)

In [ ]:
N = 32
L = 6
M = 100_000_000
%timeit generate_first_m_combinations(M, N, L)

In [ ]:
import numba as nb